# tk-assistant · Проверка окружения T4 (Этап 0)

Задача: убедиться, что **Qwen2.5-7B-Instruct в 4-bit (NF4) влезает в 16 ГБ T4** и что
QLoRA-обвязка (Unsloth + PEFT) работает. Занимает ~10 минут.

**Как запустить:** Colab → Среда выполнения → Сменить среду выполнения → **T4 GPU**, затем Runtime → Run all.

Важно к собеседованию: у T4 **нет bf16** — обучение пойдёт в **fp16** (Unsloth подберёт dtype автоматически, здесь проверим явно).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Установка Unsloth (~5 минут)Если установка сломается — свежий способ всегда в README: https://github.com/unslothai/unsloth

In [ ]:
%%capture
!pip install unsloth

## 2. Проверка CUDA и поддерживаемых точностей

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU недоступна: проверь, что выбран рантайм T4"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
bf16_ok = torch.cuda.is_bf16_supported()
print(f"bf16 поддерживается: {bf16_ok}")
assert not bf16_ok, "Это не T4 — на A100/L4 был бы bf16. Проверь выбор рантайма."
print("-> fp16 OK: T4 обучается в fp16 (ожидаемо, зафиксируй в README)")

## 3. Загрузка Qwen2.5-7B-Instruct в 4-bit NF4 — главная проверка памяти

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,   # QLoRA: 4-bit NF4
)
allocated = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nВесах модели в VRAM: {allocated:.1f} GB из {total:.1f} GB")
assert allocated < total * 0.7, "Модель заняла слишком много памяти — что-то не так"
print("OK: 7B в 4-bit влезает с запасом на активации и оптимизатор")

## 4. Подключаем LoRA с параметрами проекта и промпты из репозитория

Клонируем репо, чтобы проверить `src/prompts.py` (единый чат-шаблон) в боевых условиях.

In [ ]:
import sys, torch
!git clone https://github.com/markkentos/tk-assistant.git
sys.path.insert(0, "/content/tk-assistant/src")
from prompts import inference_prompt, SYSTEM_PROMPT
print("SYSTEM_PROMPT из src/prompts.py:\n", SYSTEM_PROMPT[:200], "...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05,          # configs/train_qlora.yaml
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
trainable, total_params = model.get_nb_trainable_parameters()
print(f"\nОбучаемых параметров: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)")

## 5. Пробная генерация в формате проекта

In [ ]:
FastLanguageModel.for_inference(model)

question = ("Может ли работодатель уволить работника, который подал заявление об "
            "увольнении по собственному желанию, а через неделю передумал?")
prompt = inference_prompt(tokenizer, question, fewshot=False)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"\nVRAM после генерации: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 6. Вердикт

Этап 0 закрыт, если выше: модель загрузилась, память < ~11 GB, генерация отвечает по-русски (base-модель может нарушать формат — это нормально, формат выучит QLoRA).

Фиксируем в README: GPU, fp16, расход VRAM, время загрузки/генерации.

In [ ]:
import datetime, torch

vram = torch.cuda.memory_allocated() / 1e9
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(
    "ЧЕК-ЛИСТ ЭТАПА 0 (перенести в README):\n"
    "- GPU: T4 16GB\n"
    "- Точность обучения: fp16 (bf16 недоступен на T4)\n"
    f"- VRAM под модель (4-bit): {vram:.1f} GB\n"
    f"- Проверено: {stamp}\n"
    "- [ ] генерация отвечает по-русски\n"
    "- [ ] LoRA подключилась (обучаемых ~0.5-1% параметров)"
)